In [0]:
import pandas as pd

In [0]:
df_circuit = spark.read.csv("/Volumes/gr5069/raw/f1_data/circuits.csv", header=True)

In [0]:
from pyspark.sql.functions import concat_ws, datediff, to_date, floor, col, avg, min, max, when, col, upper, substring, year
from pyspark.sql.window import Window

In [0]:
display(df_circuit)

circuitId,circuitRef,name,location,country,lat,lng,alt,url
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_International_Circuit
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,http://en.wikipedia.org/wiki/Bahrain_International_Circuit
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,http://en.wikipedia.org/wiki/Istanbul_Park
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers_Magny-Cours
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103,http://en.wikipedia.org/wiki/Hockenheimring


In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
display(df_pitstops.head(5))

raceId,driverId,stop,lap,time,duration,milliseconds
841,153,1,1,17:05:23,26.898,26898
841,30,1,1,17:05:52,25.021,25021
841,17,1,11,17:20:48,23.426,23426
841,4,1,12,17:22:34,23.251,23251
841,13,1,13,17:24:10,23.842,23842


In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)

In [0]:
df = df_laptimes.withColumn("milliseconds", col("milliseconds").cast("int"))
driver_avg = df.groupBy("raceId", "driverId").agg(
    floor(avg("milliseconds")).alias("avg_pit_time"),
).orderBy("raceId")

display(driver_avg.head(5))

raceId,driverId,avg_pit_time
1,7,97622
1,1,97563
1,13,97002
1,18,97513
1,4,97597


In [0]:
race_stats = df.groupBy("raceId").agg(
    min("milliseconds").alias("fastest_pit_stop in ms"),
    max("milliseconds").alias("slowest_pit_stop in ms"),
).orderBy("raceId")

display(race_stats.head(5))

raceId,fastest_pit_stop in ms,slowest_pit_stop in ms
1,87706,185713
10,81931,151967
100,78739,156583
1000,80012,106822
1001,106286,524604


In [0]:
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
drivers_pitstops = driver_avg.join(
    drivers.select("driverId", "forename", "surname"),
    on="driverId"
).orderBy("raceId")


display(drivers_pitstops.head(5))

driverId,raceId,avg_pit_time,forename,surname
7,1,97622,Sébastien,Bourdais
1,1,97563,Lewis,Hamilton
13,1,97002,Felipe,Massa
18,1,97513,Jenson,Button
4,1,97597,Fernando,Alonso


In [0]:
driver_standing = spark.read.csv("/Volumes/gr5069/raw/f1_data/driver_standings.csv", header=True)
display(driver_standing.head(5))

driverStandingsId,raceId,driverId,points,position,positionText,wins
1,18,1,10,1,1,1
2,18,2,8,2,2,0
3,18,3,6,3,3,0
4,18,4,5,4,4,0
5,18,5,4,5,5,0


In [0]:
pitstop_position = drivers_pitstops.join(
    driver_standing.select("raceId", "driverId", "position"),
    on=["raceId", "driverId"]
).withColumn("position", col("position").cast("int")).orderBy( "position", "avg_pit_time")

display(pitstop_position.head(5))

raceId,driverId,avg_pit_time,forename,surname,position
977,20,69143,Sebastian,Vettel,1
997,20,69284,Sebastian,Vettel,1
1058,830,69562,Max,Verstappen,1
1018,1,69642,Lewis,Hamilton,1
1032,822,70202,Valtteri,Bottas,1


In [0]:
race_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
display(race_results.head(5))

resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
1,18,1,1,22,1,1,1,1,10,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
2,18,2,2,3,5,2,2,2,8,58,+5.478,5696094,41,3,1:27.739,217.586,1
3,18,3,3,7,7,3,3,3,6,58,+8.163,5698779,41,5,1:28.090,216.719,1
4,18,4,4,5,11,4,4,4,5,58,+17.181,5707797,58,7,1:28.603,215.464,1
5,18,5,1,23,3,5,5,5,4,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [0]:
race_results_renamed = race_results.withColumnRenamed("position", "finish_position")

joined = pitstop_position.join(
    race_results_renamed,
    on=["raceId", "driverId"],
    how="inner"
)

Perform a train/test split.

In [0]:
df = pitstop_position.toPandas()
X_tr, X_te, y_tr, y_te = train_test_split(df.drop(["position"], axis=1), df[["position"]].values.ravel(), random_state=42)

In [0]:
pip install mlflow

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install typing_extensions --upgrade

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


1. Build any model of your choice with tunable hyperparameters

In [0]:

import mlflow.sklearn


from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

with mlflow.start_run(run_name="Basic RF Experiment") as run:
  # Create model, train it, and create predictions
  rf = RandomForestRegressor()
  rf.fit(X_train, y_train.astype(float))
  predictions = rf.predict(X_test)
  
  # Log model
  mlflow.sklearn.log_model(rf, "random-forest-model")
  
  # Create metrics
  mse = mean_squared_error(y_test, predictions)
  print("  mse: {}".format(mse))
  
  # Log metrics
  mlflow.log_metric("mse", mse)
  
  runID = run.info.run_id
  experimentID = run.info.experiment_id
  
  print("Inside MLflow Run with run_id {} and experiment_id {}".format(runID, experimentID))



 2. Create an experiment setup where - for each run - you log:

the hyperparameters used in the model
the model itself
every possible metric from the model you chose
at least two artifacts (plots, or csv files)

In [0]:
def log_rf(experimentID, run_name, params, X_train, X_test, y_train, y_test):
    import matplotlib.pyplot as plt
    import mlflow.sklearn                                                                             
    import pandas as pd
    import seaborn as sns                                                                             
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score                     
    import tempfile
                                                                                                      
    y_train = y_train.astype(float).squeeze()                                                         
    y_test  = y_test.astype(float).squeeze()
                                                                                                      
    with mlflow.start_run(experiment_id=experimentID, run_name=run_name) as run:                      
      rf = RandomForestRegressor(**params)
      rf.fit(X_train, y_train)                                                                        
      predictions = rf.predict(X_test)

      mlflow.sklearn.log_model(rf, "random-forest-model")                                             
  
      [mlflow.log_param(param, value) for param, value in params.items()]                             
                  
      mse = mean_squared_error(y_test, predictions)                                                   
      mae = mean_absolute_error(y_test, predictions)
      r2  = r2_score(y_test, predictions)                                                             
      print("  mse: {}".format(mse))
      print("  mae: {}".format(mae))
      print("  R2: {}".format(r2))                                                                    
  
      mlflow.log_metric("mse", mse)                                                                   
      mlflow.log_metric("mae", mae)
      mlflow.log_metric("r2",  r2)                                                                    
  
      # Feature importance — use the trained columns, not the original df                             
      importance = (
        pd.DataFrame(list(zip(X_train.columns, rf.feature_importances_)),                             
                     columns=["Feature", "Importance"])
          .sort_values("Importance", ascending=False)                                                 
      )
                                                                                                      
      temp = tempfile.NamedTemporaryFile(prefix="feature-importance-", suffix=".csv")                 
      try:
        importance.to_csv(temp.name, index=False)                                                     
        mlflow.log_artifact(temp.name, "feature-importance.csv")
      finally:
        temp.close()

      fig, ax = plt.subplots()
      sns.residplot(x=predictions, y=y_test, lowess=True)
      plt.xlabel("Predicted values for Position")
      plt.ylabel("Residual")                                                                          
      plt.title("Residual Plot")
                                                                                                      
      temp = tempfile.NamedTemporaryFile(prefix="residuals-", suffix=".png")                          
      try:
        fig.savefig(temp.name)                                                                        
        mlflow.log_artifact(temp.name, "residuals.png")
      finally:
        temp.close()
                                                                                                      
      display(fig)
      return run.info.run_id                                                                          
                  


Run with new parameters.

the hyperparameters used in the model
the model itself
every possible metric from the model you chose
at least two artifacts

10 Experiments

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 5,
    "random_state": 42
  }
log_rf(experimentID, "First Run", params, X_tr, X_te, y_tr, y_te)

Check the UI to see how this appears.  Take a look at the artifact to see where the plot was saved.

Now, run a third run.

In [0]:

 params={                                                                                          
    "n_estimators": 1000,
    "max_depth": 5,
    "random_state": 42
  }
log_rf(experimentID, "Second Run", params, X_tr, X_te, y_tr, y_te)

In [0]:

 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 10,
    "random_state": 42
  }
log_rf(experimentID, "Thrid Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 1000,
    "max_depth": 10,
    "random_state": 42
  }
log_rf(experimentID, "Fourth Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 15,
    "random_state": 42
  }
log_rf(experimentID, "Fifth Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 50,
    "max_depth": 10,
    "random_state": 42
  }
log_rf(experimentID, "Sixth Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 20,
    "random_state": 42
  }
log_rf(experimentID, "Seventh Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 30,
    "random_state": 42
  }
log_rf(experimentID, "Eighth Run", params, X_tr, X_te, y_tr, y_te)

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 75,
    "random_state": 42
  }
log_rf(experimentID, "Nineth Run", params, X_tr, X_te, y_tr, y_te)

Select your best model

The 10th experiment is the best model. it has the highest R square, and the the lowest MSE. The Rsquare is still not high enough. Potential improvement can be improve the dataset and features. The model has a really high max depth now, and has issue of overfitting. 

 mse: 22.761748048980355
  mae: 3.4836312673443275
  R2: 0.44808579274339166

In [0]:
 params={                                                                                          
    "n_estimators": 100,
    "max_depth": 100,
    "random_state": 42
  }
log_rf(experimentID, "Tenth Run", params, X_tr, X_te, y_tr, y_te)